# Lab 8: Implementation and Performance Evaluation of Categorical Naive Bayes Classifier

**Dataset:** Play Tennis (Classic Decision-Tree Benchmark)  
**Source:** `datasets/tennis.csv`  
**Reg No:** 2547237

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

# Aesthetic settings
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.family': 'DejaVu Sans'
})

RANDOM_STATE = 42
print('Libraries loaded successfully.')

**Interpretation:**  
We import the essential libraries:
- `pandas` / `numpy` for data loading and manipulation.
- `matplotlib` / `seaborn` for rich visualisations.
- `LabelEncoder` to convert categorical strings into integer codes required by `CategoricalNB`.
- `CategoricalNB`, `DecisionTreeClassifier`, `LogisticRegression`, and `SVC` are the four classifiers we will train and compare.
- Evaluation helpers (`accuracy_score`, `confusion_matrix`, `classification_report`, `ConfusionMatrixDisplay`) to quantify model performance.

---
## 2. Data Loading & Exploration

In [ ]:
# Load dataset
df = pd.read_csv('../datasets/tennis.csv')

# Drop the row-number column 'No'
df.drop(columns=['No'], inplace=True)

# Rename target column for convenience
df.rename(columns={'Play Tennis': 'Play'}, inplace=True)

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nClass distribution:')
print(df['Play'].value_counts())

**Interpretation:**  
The dataset contains **50 instances** (an augmented version of the classic 14-row Play Tennis table) and **4 categorical input features**: Outlook, Temperature, Humidity, and Wind. The target variable `Play` takes values *Yes* or *No*. There are no missing values, so no imputation is required. The class distribution confirms a moderate class imbalance with more *Yes* labels than *No* labels.

---
## 3. Exploratory Data Analysis & Visualisation

In [ ]:
features = ['Outlook', 'Temperature', 'Humidity', 'Wind']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Feature Value Distributions by Play Decision', fontsize=16, fontweight='bold', y=1.01)

palette = {'Yes': '#2ecc71', 'No': '#e74c3c'}

for ax, feat in zip(axes.ravel(), features):
    counts = df.groupby([feat, 'Play']).size().unstack(fill_value=0)
    counts.plot(kind='bar', ax=ax, color=[palette['No'], palette['Yes']],
                edgecolor='white', linewidth=0.8, rot=0)
    ax.set_title(f'{feat} vs Play', fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')
    ax.legend(title='Play', labels=['No', 'Yes'])
    for bar in ax.patches:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.15,
                    str(int(h)), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

**Interpretation:**  
- **Outlook:** *Overcast* days always result in playing. *Sunny* and *Rain* days are mixed — Sunny tends toward *No*, while Rain skews *Yes*.
- **Temperature:** All three temperature values have both *Yes* and *No* outcomes; *Hot* weather tilts slightly toward *No*.
- **Humidity:** *High* humidity leans toward *No*; *Normal* humidity strongly favours *Yes*.
- **Wind:** *Weak* wind almost always leads to *Yes*; *Strong* wind is more evenly split.

In [ ]:
# Stacked percentage bar chart — proportion of Play within each Outlook
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution pie
play_counts = df['Play'].value_counts()
axes[0].pie(
    play_counts,
    labels=play_counts.index,
    autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'],
    startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[0].set_title('Overall Class Distribution (Play Tennis)', fontweight='bold')

# Stacked 100% bar for Outlook
outlook_play = df.groupby(['Outlook', 'Play']).size().unstack(fill_value=0)
outlook_play_pct = outlook_play.div(outlook_play.sum(axis=1), axis=0) * 100
outlook_play_pct.plot(
    kind='bar', stacked=True, ax=axes[1],
    color=['#e74c3c', '#2ecc71'], edgecolor='white', rot=0
)
axes[1].set_title('Outlook → Play (Stacked %)', fontweight='bold')
axes[1].set_xlabel('Outlook')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Play')
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

**Interpretation:**  
The pie chart shows that roughly **64% of days** resulted in playing tennis and **36% did not**, confirming a mild class imbalance. The stacked bar chart reinforces that *Overcast* is 100% *Yes*, making it the most decisive feature value.

In [ ]:
# Heatmap of feature co-occurrence with Play (label-encoded)
df_num = df.copy()
le_temp = LabelEncoder()
for col in df_num.columns:
    df_num[col] = le_temp.fit_transform(df_num[col])

plt.figure(figsize=(8, 5))
corr = df_num.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap (Label-Encoded)', fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:**  
Because label-encoding of nominal categories assigns arbitrary integer codes, Pearson correlation should be interpreted cautiously here; it serves as a rough ordinal proxy. Humidity shows the strongest negative correlation with Play (higher encoded humidity → less likely to play), while Outlook has a moderate positive correlation. Temperature and Wind show weaker linear relationships with the target.

---
## 4. Data Preprocessing — Label Encoding

In [ ]:
# ---------------------------------------------------------------
# Separate features and target
# ---------------------------------------------------------------
feature_cols = ['Outlook', 'Temperature', 'Humidity', 'Wind']
target_col   = 'Play'

X_raw = df[feature_cols].copy()
y_raw = df[target_col].copy()

# ---------------------------------------------------------------
# Apply LabelEncoder to every feature column independently
# CategoricalNB requires non-negative integer inputs.
# ---------------------------------------------------------------
encoders = {}   # store each encoder so we can transform the query later
X_enc = X_raw.copy()

for col in feature_cols:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X_raw[col])
    encoders[col] = le
    print(f'{col:12s} → classes: {list(le.classes_)} → codes: {list(range(len(le.classes_)))}')

# Encode target
le_target = LabelEncoder()
y_enc = le_target.fit_transform(y_raw)
print(f'\nTarget       → classes: {list(le_target.classes_)} → codes: {list(range(len(le_target.classes_)))}')

print('\nEncoded feature matrix (first 5 rows):')
print(X_enc.head())

**Interpretation:**  
Each categorical column is independently label-encoded into integer codes starting from 0. We save the fitted `LabelEncoder` objects in the `encoders` dict — this lets us transform the single-sample query (Step 6) using the exact same mapping seen during training, avoiding inconsistencies. The target `Play` is similarly encoded as 0 = *No*, 1 = *Yes*.

In [ ]:
# Visual confirmation — encoded values side by side with original
comparison = pd.concat(
    [X_raw.reset_index(drop=True), X_enc.reset_index(drop=True)],
    axis=1,
    keys=['Original', 'Encoded']
)
comparison.head(10)

---
## 5. Dataset Partitioning — 80 : 20 Train-Test Split

In [ ]:
X = X_enc.values   # numpy array for sklearn compatibility
y = y_enc

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y          # preserve class proportions in each split
)

print(f'Total samples   : {len(X)}')
print(f'Training samples: {len(X_train)}  ({len(X_train)/len(X)*100:.0f}%)')
print(f'Testing samples : {len(X_test)}   ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nTrain class distribution: No={sum(y_train==0)}, Yes={sum(y_train==1)}')
print(f'Test  class distribution: No={sum(y_test==0)},  Yes={sum(y_test==1)}')

**Interpretation:**  
We use an 80:20 split with `stratify=y` to ensure both the training and test sets reflect the same class ratio as the full dataset. This prevents the test set from accidentally containing only one class, which would produce misleadingly high accuracy on such a small dataset.

---
## 6. Naive Bayes Model — Training & Evaluation

In [ ]:
# ---------------------------------------------------------------
# Train CategoricalNB
# alpha=1.0 → Laplace smoothing to handle unseen category combos
# ---------------------------------------------------------------
nb_model = CategoricalNB(alpha=1.0)
nb_model.fit(X_train, y_train)

# Predictions on the test set
y_pred_nb = nb_model.predict(X_test)

nb_accuracy = accuracy_score(y_test, y_pred_nb)
print('=' * 50)
print('       CATEGORICAL NAIVE BAYES EVALUATION')
print('=' * 50)
print(f'\nModel Accuracy : {nb_accuracy * 100:.2f}%\n')

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_nb))

print('\nClassification Report:')
print(classification_report(
    y_test, y_pred_nb,
    target_names=le_target.classes_
))

In [ ]:
# Visual confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred_nb),
    display_labels=le_target.classes_
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Categorical Naive Bayes', fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:**  
The Categorical Naive Bayes model correctly classifies the majority of test instances. True Positives (TP) and True Negatives (TN) populate the main diagonal of the confusion matrix; any off-diagonal entries represent misclassifications. Precision, Recall, and F1-Score per class help us understand performance beyond raw accuracy — especially important when one class (e.g. *Yes*) is more frequent than the other.

---
## 7. Single-Sample Inference — Custom Query

In [ ]:
# ---------------------------------------------------------------
# Query: Outlook=Sunny, Temperature=Cool, Humidity=High, Wind=Strong
# ---------------------------------------------------------------
query = {
    'Outlook'    : 'Sunny',
    'Temperature': 'Cool',
    'Humidity'   : 'High',
    'Wind'       : 'Strong'
}

# Transform query using the saved LabelEncoders (same mapping as training)
query_enc = np.array([[encoders[col].transform([query[col]])[0] for col in feature_cols]])

print('Query (original):', query)
print('Query (encoded) :', query_enc)

# Predict class label
query_pred      = nb_model.predict(query_enc)[0]
query_proba     = nb_model.predict_proba(query_enc)[0]
predicted_label = le_target.inverse_transform([query_pred])[0]

print('\n' + '=' * 45)
print('  SINGLE-SAMPLE INFERENCE (Naive Bayes)')
print('=' * 45)
print(f'  Predicted class : {predicted_label}')
for cls, prob in zip(le_target.classes_, query_proba):
    bar = '█' * int(prob * 30)
    print(f'  P({cls:3s}) = {prob:.4f}  {bar}')
print('=' * 45)

In [ ]:
# Probability bar chart for the query
fig, ax = plt.subplots(figsize=(6, 3.5))
bar_colors = ['#e74c3c', '#2ecc71']
bars = ax.barh(le_target.classes_, query_proba, color=bar_colors,
               edgecolor='white', linewidth=1.2, height=0.4)

for bar, prob in zip(bars, query_proba):
    ax.text(prob + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{prob:.4f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 1.15)
ax.set_xlabel('Probability')
ax.set_title(
    'NB Predicted Probabilities\n'
    'Query: Sunny | Cool | High | Strong',
    fontweight='bold'
)
ax.axvline(0.5, color='grey', linestyle='--', linewidth=0.8, label='Decision boundary')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:**  
For the query `{Outlook: Sunny, Temperature: Cool, Humidity: High, Wind: Strong}`, the Naive Bayes model computes the posterior probability of each class by multiplying the class prior with the product of per-feature likelihoods (conditional independence assumption). The class with the higher posterior is chosen as the prediction. Laplace smoothing (α = 1.0) ensures no zero-probability issues even if a feature-value/class combination was unseen in training.

---
## 8. Model Comparison — Decision Tree, Logistic Regression & SVM

In [ ]:
# ---------------------------------------------------------------
# Instantiate the three additional classifiers
# ---------------------------------------------------------------
dt_model  = DecisionTreeClassifier(random_state=RANDOM_STATE)
lr_model  = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
svm_model = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)

models = {
    'Categorical NB'    : nb_model,
    'Decision Tree'     : dt_model,
    'Logistic Regression': lr_model,
    'SVM (RBF)'         : svm_model
}

# Train the three new models (NB is already trained)
for name, mdl in models.items():
    if name != 'Categorical NB':
        mdl.fit(X_train, y_train)
    print(f'{name} — trained.')

In [ ]:
# ---------------------------------------------------------------
# Collect metrics + query predictions for all models
# ---------------------------------------------------------------
results = []

for name, mdl in models.items():
    y_pred   = mdl.predict(X_test)
    acc      = accuracy_score(y_test, y_pred) * 100
    q_pred   = mdl.predict(query_enc)[0]
    q_label  = le_target.inverse_transform([q_pred])[0]
    q_proba  = mdl.predict_proba(query_enc)[0]   # [P(No), P(Yes)]
    results.append({
        'Model'              : name,
        'Test Accuracy (%)'  : round(acc, 2),
        'Query Prediction'   : q_label,
        'P(No)'              : round(q_proba[0], 4),
        'P(Yes)'             : round(q_proba[1], 4)
    })

results_df = pd.DataFrame(results).set_index('Model')

print('=' * 72)
print('                   MODEL COMPARISON TABLE')
print('=' * 72)
print(results_df.to_string())
print('=' * 72)

In [ ]:
# ---------------------------------------------------------------
# Visual comparison — accuracy bar chart + probability grouped bars
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
model_names = results_df.index.tolist()
acc_vals    = results_df['Test Accuracy (%)'].values

# -- Accuracy bar --
palette_acc = ['#3498db', '#e67e22', '#9b59b6', '#1abc9c']
bars = axes[0].bar(model_names, acc_vals, color=palette_acc,
                   edgecolor='white', linewidth=1.2, width=0.5)
axes[0].set_ylim(0, 115)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Test Accuracy — All Models', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
for bar, acc in zip(bars, acc_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
                 f'{acc:.1f}%', ha='center', fontsize=10, fontweight='bold')

# -- Query probability grouped bar --
x = np.arange(len(model_names))
width = 0.35
p_no  = results_df['P(No)'].values
p_yes = results_df['P(Yes)'].values

b1 = axes[1].bar(x - width/2, p_no,  width, label='P(No)',  color='#e74c3c', edgecolor='white')
b2 = axes[1].bar(x + width/2, p_yes, width, label='P(Yes)', color='#2ecc71', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=15, ha='right')
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Probability')
axes[1].set_title('Query Probabilities — Sunny|Cool|High|Strong', fontweight='bold')
axes[1].axhline(0.5, color='grey', linestyle='--', linewidth=0.8)
axes[1].legend()

for bar in list(b1) + list(b2):
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width() / 2, h + 0.02,
                 f'{h:.2f}', ha='center', fontsize=8)

plt.suptitle('Model Comparison Dashboard', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Per-model confusion matrices in a 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
cmaps = ['Blues', 'Oranges', 'Purples', 'Greens']

for ax, (name, mdl), cmap in zip(axes.ravel(), models.items(), cmaps):
    y_pred = mdl.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=le_target.classes_).plot(
        ax=ax, cmap=cmap, colorbar=False
    )
    ax.set_title(f'{name}', fontweight='bold')

fig.suptitle('Confusion Matrices — All Classifiers', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:**  
The confusion matrices for all four classifiers show how each model partitions the test set. Models with more entries on the main diagonal are better classifiers. On this small dataset, all models tend to perform similarly in raw accuracy, but may differ on which specific test instances they misclassify.

In [ ]:
# ---------------------------------------------------------------
# Full classification reports side by side
# ---------------------------------------------------------------
print('\n' + '=' * 60)
for name, mdl in models.items():
    y_pred = mdl.predict(X_test)
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=le_target.classes_))
print('=' * 60)

---
## 9. Analysis Report

### Why do models produce different predictions and probabilities for the same query?

1. **Categorical Naive Bayes** computes the posterior probability using Bayes' theorem under a strict **conditional independence** assumption — each feature contributes multiplicatively. For the query `{Sunny, Cool, High, Strong}`, Sunny and High-humidity both carry strong *No* evidence, so their probabilities compound, driving a confident *No* prediction.

2. **Decision Tree** partitions the feature space by choosing the most informative split at each node (via Gini impurity or information gain). It may return probability estimates of 0 or 1 for queries that fall into a pure leaf node, reflecting hard, rule-based logic rather than probabilistic smoothing.

3. **Logistic Regression** models the log-odds of each class as a **linear combination** of the encoded features. Because it optimises a global loss function, it balances all training samples and the probability it assigns is a smooth sigmoid function — generally less extreme than a Decision Tree but influenced by the numerical magnitude of the encoded values.

4. **SVM (RBF kernel)** finds the **maximum-margin hyperplane** in a high-dimensional kernel space; probabilities are estimated via Platt scaling on top of the decision function. Because SVM optimises geometric separation rather than likelihood, its probability estimates can differ substantially from models trained directly under probabilistic frameworks — especially on small datasets where the margin geometry is sensitive to individual support vectors.

---
## 10. Summary Table

In [ ]:
print('\nFINAL RESULTS SUMMARY')
print('Query: Outlook=Sunny | Temperature=Cool | Humidity=High | Wind=Strong')
print()
print(results_df.to_string())